# Sync Odoo and Bexio: Contacts & Products

Bidirectional sync of contacts and products between Odoo (via JSON-2 API) and Bexio (via REST API).
On first run a full sync is performed; subsequent runs use delta sync based on `updated_at` / `write_date`.


## Setup

Load credentials from `.env`, create SSL context, and define helpers for Odoo JSON-2 and Bexio REST calls.

In [ ]:
import json
import ssl
import urllib.request
import urllib.error
from pathlib import Path
from datetime import datetime, timezone

# ------------------------------------------------------------------
# Load credentials from .env
# ------------------------------------------------------------------
env_path = Path('.env')
ODOO_API_TOKEN = ''
BEXIO_API_TOKEN = ''
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line.startswith('ODOO_API_TOKEN='):
            ODOO_API_TOKEN = line.split('=', 1)[1]
        elif line.startswith('BEXIO_API_TOKEN='):
            BEXIO_API_TOKEN = line.split('=', 1)[1]

ODOO_URL = 'https://jupyter-lab.19.odoo.build'
ODOO_DB  = 'jupyter-lab'
BEXIO_API_URL = 'https://api.bexio.com/2.0'

ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode = ssl.CERT_NONE

# File to remember last-sync timestamps for delta sync
LAST_SYNC_FILE = Path('tmp/last_sync.json')
LAST_SYNC_FILE.parent.mkdir(parents=True, exist_ok=True)

def load_last_sync():
    if LAST_SYNC_FILE.exists():
        return json.loads(LAST_SYNC_FILE.read_text())
    return {}

def save_last_sync(timestamps):
    LAST_SYNC_FILE.write_text(json.dumps(timestamps, indent=2, default=str))

last_sync = load_last_sync()
print('Setup complete. Last sync timestamps:', last_sync)


## API Helpers

In [ ]:
# ------------------------------------------------------------------
# Odoo JSON-2 helper
# ------------------------------------------------------------------
def odoo_json2(model, method, params):
    url = f"{ODOO_URL.rstrip('/')}/json/2/{model}/{method}"
    payload = json.dumps(params).encode()
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {ODOO_API_TOKEN}',
    }
    req = urllib.request.Request(url, data=payload, headers=headers)
    try:
        with urllib.request.urlopen(req, context=ssl_ctx) as resp:
            return json.loads(resp.read())
    except urllib.error.HTTPError as e:
        raise Exception(f'Odoo HTTP {e.code}: {e.read().decode()}')

# ------------------------------------------------------------------
# Bexio REST helper
# ------------------------------------------------------------------
def bexio_request(endpoint, method='GET', data=None):
    url = f"{BEXIO_API_URL.rstrip('/')}/{endpoint.lstrip('/')}"
    headers = {
        'Accept': 'application/json',
        'Authorization': f'Bearer {BEXIO_API_TOKEN}',
    }
    if data is not None:
        headers['Content-Type'] = 'application/json'
        payload = json.dumps(data).encode()
    else:
        payload = None
    req = urllib.request.Request(url, data=payload, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req, context=ssl_ctx) as resp:
            return json.loads(resp.read())
    except urllib.error.HTTPError as e:
        raise Exception(f'Bexio HTTP {e.code}: {e.read().decode()}')

print('Helpers defined.')


## Test Connections

In [ ]:
# Odoo
odoo_count = odoo_json2('res.partner', 'search_count', {'domain': []})
print(f'Odoo partners: {odoo_count}')

# Bexio
bexio_contacts_raw = bexio_request('/contact')
print(f'Bexio contacts: {len(bexio_contacts_raw)}')


## Field Mapping

Mapping between Bexio `contact` fields and Odoo `res.partner` fields.

| Bexio field | Odoo field | Notes |
|-------------|------------|-------|
| `id` | `bexio_id` (stored as XML-ID) | Unique Bexio identifier |
| `name_1` | `name` | Company name or first name |
| `name_2` | `name` | Appended for persons (`name_1 name_2`) |
| `mail` | `email` | Primary e-mail address |
| `phone_fixed` | `phone` | Landline number |
| `phone_mobile` | `phone` | Mobile number (Odoo has no `mobile` field in this build) |
| `street_name` | `street` | Writable street field in Bexio |
| `address` | `street` | Read-only computed address in Bexio |
| `postcode` | `zip` | ZIP / postal code |
| `city` | `city` | City name |
| `country_id` | `country_id` | Bexio country ID ↔ Odoo country many2one |
| `contact_type_id` | `company_type` / `is_company` | `1` = company, `2` = person |
| `updated_at` | `write_date` | Used for delta sync |
| `nr` | `ref` | Bexio contact number || `remarks` | `comment` | Remarks / internal notes |


## Fetch Odoo Partners

In [ ]:
ODOO_FIELDS = [
    'id', 'name', 'email', 'phone',
    'street', 'zip', 'city', 'country_id',
    'company_type', 'is_company', 'write_date', 'ref',
    'comment'
]

odoo_ids = odoo_json2('res.partner', 'search', {'domain': []})
print(f'Fetched {len(odoo_ids)} Odoo partner(s).')

odoo_partners = []
BATCH = 200
for i in range(0, len(odoo_ids), BATCH):
    batch = odoo_ids[i:i + BATCH]
    odoo_partners.extend(odoo_json2('res.partner', 'read', {'ids': batch, 'fields': ODOO_FIELDS}))

# Build lookup indexes
odoo_by_email = {}
odoo_by_name = {}
for p in odoo_partners:
    email = (p.get('email') or '').strip().lower()
    if email:
        odoo_by_email[email] = p
    name = (p.get('name') or '').strip().lower()
    if name:
        odoo_by_name[name] = p

print(f'Indexed {len(odoo_by_email)} by e-mail, {len(odoo_by_name)} by name.')


## Fetch Bexio Contacts

In [ ]:
# Bexio does not expose a queryable updated_at filter on the standard list endpoint,
# so we fetch all contacts and filter locally.
bexio_raw = bexio_request('/contact')
print(f'Fetched {len(bexio_raw)} Bexio contact(s).')

if last_sync.get('bexio'):
    cutoff = last_sync['bexio']
    bexio_contacts = [c for c in bexio_raw if c.get('updated_at') and c['updated_at'] > cutoff]
    print(f"Delta sync: {len(bexio_contacts)} Bexio contact(s) updated after {cutoff}")
else:
    bexio_contacts = bexio_raw
    print('Full sync: using all Bexio contacts.')

# Build lookup indexes
bexio_by_email = {}
bexio_by_name = {}
for c in bexio_contacts:
    email = (c.get('mail') or '').strip().lower()
    if email:
        bexio_by_email[email] = c
    name = ' '.join(p for p in [c.get('name_1', ''), c.get('name_2', '')] if p).strip().lower()
    if name:
        bexio_by_name[name] = c

print(f'Indexed {len(bexio_by_email)} by e-mail, {len(bexio_by_name)} by name.')


## Cache Country Mapping

Bexio stores `country_id` as an integer; Odoo uses a many2one.
We build bidirectional lookups via ISO-3166 alpha2 codes.

In [ ]:
# 1. Fetch Bexio countries
bexio_countries = bexio_request('/country')
print(f'Loaded {len(bexio_countries)} Bexio countries.')

# 2. Fetch Odoo countries
odoo_country_ids = odoo_json2('res.country', 'search', {'domain': []})
odoo_countries = odoo_json2('res.country', 'read', {'ids': odoo_country_ids, 'fields': ['id', 'code']})
odoo_country_by_code = {c['code'].upper(): c['id'] for c in odoo_countries if c.get('code')}
print(f'Loaded {len(odoo_countries)} Odoo countries.')

# 3. Build Bexio country_id → Odoo country_id map
country_map = {}
reverse_country_map = {}
for bc in bexio_countries:
    code = (bc.get('iso_3166_alpha2') or '').upper()
    if code and code in odoo_country_by_code:
        country_map[bc['id']] = odoo_country_by_code[code]
        reverse_country_map[odoo_country_by_code[code]] = bc['id']

print(f'Mapped {len(country_map)} countries.')


## XML-ID Helper

Store the Bexio contact ID as an Odoo external identifier (`ir.model.data`).

In [ ]:
def set_bexio_xml_id(partner_id, bexio_id):
    """Create or update the Bexio XML-ID for a partner."""
    existing = odoo_json2('ir.model.data', 'search', {
        'domain': [
            ('model', '=', 'res.partner'),
            ('res_id', '=', partner_id),
            ('module', '=', '__sync__')
        ]
    })
    target_name = f'bexio_contact_{bexio_id}'
    if existing:
        current = odoo_json2('ir.model.data', 'read', {'ids': existing, 'fields': ['name']})[0]
        if current['name'] != target_name:
            odoo_json2('ir.model.data', 'write', {'ids': existing, 'vals': {'name': target_name}})
    else:
        odoo_json2('ir.model.data', 'create', {'vals_list': [{
            'name': target_name,
            'model': 'res.partner',
            'res_id': partner_id,
            'module': '__sync__'
        }]})
    return f'__sync__.{target_name}'

print('XML-ID helpers defined.')


## Sync Logic

1. **Bexio → Odoo** — create or update Odoo partners from Bexio contacts.
2. **Odoo → Bexio** — update Bexio contacts from Odoo partners.

Matching strategy: **e-mail address** preferred; falls back to **name**.

In [ ]:
# ------------------------------------------------------------------
# Transform helpers
# ------------------------------------------------------------------
def bexio_to_odoo_vals(contact):
    """Convert a Bexio contact dict to Odoo res.partner write values."""
    vals = {}
    name_parts = [contact.get('name_1', ''), contact.get('name_2', '')]
    name = ' '.join(p for p in name_parts if p).strip()
    if name:
        vals['name'] = name
    email = contact.get('mail')
    if email:
        vals['email'] = email.strip()
    phone = contact.get('phone_fixed') or contact.get('phone_mobile')
    if phone:
        vals['phone'] = phone
    if contact.get('address'):
        vals['street'] = contact['address']
    elif contact.get('street_name'):
        vals['street'] = contact['street_name']
    if contact.get('postcode'):
        vals['zip'] = contact['postcode']
    if contact.get('city'):
        vals['city'] = contact['city']
    if contact.get('country_id') and contact['country_id'] in country_map:
        vals['country_id'] = country_map[contact['country_id']]
    ctype = contact.get('contact_type_id')
    if ctype == 1:
        vals['company_type'] = 'company'
        vals['is_company'] = True
    elif ctype == 2:
        vals['company_type'] = 'person'
        vals['is_company'] = False
    if contact.get('nr'):
        vals['ref'] = str(contact['nr'])
    if contact.get('remarks'):
        vals['comment'] = contact['remarks']
    return vals

def odoo_to_bexio_vals(partner, for_create=False):
    """Convert an Odoo partner dict to Bexio contact write values."""
    vals = {}
    if partner.get('name'):
        names = partner['name'].split(' ', 1)
        vals['name_1'] = names[0]
        if len(names) > 1:
            vals['name_2'] = names[1]
    if partner.get('email'):
        vals['mail'] = partner['email'].strip()
    if partner.get('phone'):
        vals['phone_fixed'] = partner['phone']
    if partner.get('street'):
        vals['street_name'] = partner['street']
    if partner.get('zip'):
        vals['postcode'] = partner['zip']
    if partner.get('city'):
        vals['city'] = partner['city']
    # Map Odoo country_id back to Bexio country_id
    country = partner.get('country_id')
    if isinstance(country, list) and len(country) == 2:
        country = country[0]
    if country and country in reverse_country_map:
        vals['country_id'] = reverse_country_map[country]
    if partner.get('company_type') == 'company':
        vals['contact_type_id'] = 1
    else:
        vals['contact_type_id'] = 2
    if partner.get('comment'):
        vals['remarks'] = partner['comment']
    if for_create:
        vals['user_id'] = 1
        vals['owner_id'] = 1
    return vals

print('Transform helpers defined.')


### Bexio → Odoo

In [ ]:
created_odoo = 0
updated_odoo = 0
skipped_odoo = 0

for contact in bexio_contacts:
    vals = bexio_to_odoo_vals(contact)
    if not vals.get('name'):
        skipped_odoo += 1
        continue

    email = (contact.get('mail') or '').strip().lower()
    name = vals['name'].lower()

    # Try e-mail match first, then name match
    partner = odoo_by_email.get(email) if email else None
    if not partner:
        partner = odoo_by_name.get(name)

    if partner:
        odoo_json2('res.partner', 'write', {'ids': [partner['id']], 'vals': vals})
        updated_odoo += 1
        set_bexio_xml_id(partner['id'], contact['id'])
    else:
        new_ids = odoo_json2('res.partner', 'create', {'vals_list': [vals]})
        new_id = new_ids[0] if isinstance(new_ids, list) else new_ids
        created_odoo += 1
        set_bexio_xml_id(new_id, contact['id'])
        # Update local indexes so Odoo → Bexio can skip it
        vals['id'] = new_id
        if vals.get('email'):
            odoo_by_email[vals['email'].lower()] = vals
        odoo_by_name[name] = vals

print(f'Bexio → Odoo: {created_odoo} created, {updated_odoo} updated, {skipped_odoo} skipped.')


### Odoo → Bexio

In [ ]:
created_bexio = 0
updated_bexio = 0
skipped_bexio = 0

# Re-fetch Bexio contacts to include any created in previous steps
bexio_raw_fresh = bexio_request('/contact')
bexio_fresh_by_email = {}
bexio_fresh_by_name = {}
for c in bexio_raw_fresh:
    em = (c.get('mail') or '').strip().lower()
    if em:
        bexio_fresh_by_email[em] = c
    nm = ' '.join(p for p in [c.get('name_1', ''), c.get('name_2', '')] if p).strip().lower()
    if nm:
        bexio_fresh_by_name[nm] = c

for partner in odoo_partners:
    # Delta sync: only push Odoo partners that changed since last sync
    if last_sync.get('odoo') and partner.get('write_date', '') <= last_sync['odoo']:
        continue

    vals = odoo_to_bexio_vals(partner, for_create=False)
    if not vals.get('name_1'):
        skipped_bexio += 1
        continue

    email = (partner.get('email') or '').strip().lower()
    name = (partner.get('name') or '').strip().lower()

    contact = bexio_fresh_by_email.get(email) if email else None
    if not contact:
        contact = bexio_fresh_by_name.get(name)

    if contact:
        bexio_request(f"/contact/{contact['id']}", method='POST', data=vals)
        updated_bexio += 1
    else:
        vals = odoo_to_bexio_vals(partner, for_create=True)
        bexio_request('/contact', method='POST', data=vals)
        created_bexio += 1

print(f'Odoo → Bexio: {created_bexio} created, {updated_bexio} updated, {skipped_bexio} skipped.')


## Persist Sync Timestamps

In [ ]:
# Record max Bexio updated_at from after the sync
bexio_raw_final = bexio_request('/contact')
bexio_timestamps = [c.get('updated_at', '') for c in bexio_raw_final if c.get('updated_at')]
last_sync['bexio'] = max(bexio_timestamps) if bexio_timestamps else ''

# Record current UTC time for Odoo
last_sync['odoo'] = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')

save_last_sync(last_sync)
print(f'Sync timestamps saved: {last_sync}')


## Sync Summary

In [ ]:
print('=' * 50)
print('SYNC SUMMARY')
print('=' * 50)
print(f"{'Direction':<20} {'Created':>8} {'Updated':>8} {'Skipped':>8}")
print('-' * 50)
print(f"{'Bexio → Odoo':<20} {created_odoo:>8} {updated_odoo:>8} {skipped_odoo:>8}")
print(f"{'Odoo → Bexio':<20} {created_bexio:>8} {updated_bexio:>8} {skipped_bexio:>8}")
print('=' * 50)
print(f"Next run will use delta sync from {last_sync}")


## Product Field Mapping

Mapping between Bexio `article` fields and Odoo `product.template` fields.

| Bexio field | Odoo field | Notes |
|-------------|------------|-------|
| `id` | `bexio_id` (stored as XML-ID) | Unique Bexio identifier |
| `intern_code` | `default_code` | Product code (used as sync key) |
| `intern_name` | `name` | Product name |
| `intern_description` | `description` | Internal description |
| `sale_price` | `list_price` | Sales price |
| `purchase_price` | `standard_price` | Cost price |
| `remarks` | `description_sale` | Remarks / sales description |
| `updated_at` | `write_date` | Used for delta sync |


## Fetch Odoo Products


In [ ]:
ODOO_PRODUCT_FIELDS = [
    'id', 'name', 'default_code', 'list_price',
    'standard_price', 'description', 'description_sale',
    'type', 'write_date', 'active'
]

odoo_product_ids = odoo_json2('product.template', 'search', {'domain': []})
print(f'Fetched {len(odoo_product_ids)} Odoo product template(s).')

odoo_products = []
BATCH = 200
for i in range(0, len(odoo_product_ids), BATCH):
    batch = odoo_product_ids[i:i + BATCH]
    odoo_products.extend(odoo_json2('product.template', 'read', {'ids': batch, 'fields': ODOO_PRODUCT_FIELDS}))

# Build lookup by product code (default_code)
odoo_by_code = {}
for p in odoo_products:
    code = (p.get('default_code') or '').strip()
    if code:
        odoo_by_code[code] = p

print(f'Indexed {len(odoo_by_code)} Odoo product(s) with a product code.')


## Fetch Bexio Articles


In [ ]:
# Fetch all Bexio articles and filter locally for delta sync
bexio_articles_raw = bexio_request('/article')
print(f'Fetched {len(bexio_articles_raw)} Bexio article(s).')

if last_sync.get('bexio_products'):
    cutoff = last_sync['bexio_products']
    bexio_articles = [a for a in bexio_articles_raw if a.get('updated_at') and a['updated_at'] > cutoff]
    print(f"Delta sync: {len(bexio_articles)} Bexio article(s) updated after {cutoff}")
else:
    bexio_articles = bexio_articles_raw
    print('Full sync: all Bexio articles will be processed.')

# Build lookup by intern_code
bexio_by_code = {}
for a in bexio_articles_raw:
    code = (a.get('intern_code') or '').strip()
    if code:
        bexio_by_code[code] = a

print(f'Indexed {len(bexio_by_code)} Bexio article(s) with an internal code.')


## Product XML-ID Helper

Store the Bexio article ID as an Odoo external identifier (`ir.model.data`).


In [ ]:
def set_bexio_product_xml_id(product_id, bexio_id):
    """Create or update the Bexio XML-ID for a product.template."""
    existing = odoo_json2('ir.model.data', 'search', {
        'domain': [
            ('model', '=', 'product.template'),
            ('res_id', '=', product_id),
            ('module', '=', '__sync__')
        ]
    })
    target_name = f'bexio_article_{bexio_id}'
    if existing:
        current = odoo_json2('ir.model.data', 'read', {'ids': existing, 'fields': ['name']})[0]
        if current['name'] != target_name:
            odoo_json2('ir.model.data', 'write', {'ids': existing, 'vals': {'name': target_name}})
    else:
        odoo_json2('ir.model.data', 'create', {'vals_list': [{
            'name': target_name,
            'model': 'product.template',
            'res_id': product_id,
            'module': '__sync__'
        }]})
    print(f'  Linked Odoo product {product_id} ↔ Bexio article {bexio_id}')


## Product Sync Logic

1. **Bexio → Odoo** — create or update Odoo products from Bexio articles.
2. **Odoo → Bexio** — update Bexio articles from Odoo products.

Matching strategy: **product code** (`intern_code` / `default_code`) exact match.


In [ ]:
# ------------------------------------------------------------------
# Product transform helpers
# ------------------------------------------------------------------
def bexio_to_odoo_product_vals(article):
    """Convert a Bexio article dict to Odoo product.template write values."""
    vals = {}
    if article.get('intern_name'):
        vals['name'] = article['intern_name']
    if article.get('intern_description'):
        vals['description'] = article['intern_description']
    if 'sale_price' in article:
        vals['list_price'] = float(article['sale_price'] or 0)
    if 'purchase_price' in article:
        vals['standard_price'] = float(article['purchase_price'] or 0)
    if article.get('remarks'):
        vals['description_sale'] = article['remarks']
    return vals


def odoo_to_bexio_product_vals(product):
    """Convert an Odoo product.template dict to Bexio article write values."""
    vals = {}
    if product.get('name'):
        vals['intern_name'] = product['name']
    if product.get('description'):
        vals['intern_description'] = product['description']
    if 'list_price' in product:
        vals['sale_price'] = float(product['list_price'] or 0)
    if 'standard_price' in product:
        vals['purchase_price'] = float(product['standard_price'] or 0)
    if product.get('description_sale'):
        vals['remarks'] = product['description_sale']
    return vals


### Bexio → Odoo (Products)


In [ ]:
created_odoo_products = 0
updated_odoo_products = 0
skipped_odoo_products = 0

for article in bexio_articles:
    code = (article.get('intern_code') or '').strip()
    if not code:
        skipped_odoo_products += 1
        continue

    vals = bexio_to_odoo_product_vals(article)
    if not vals.get('name'):
        skipped_odoo_products += 1
        continue

    product = odoo_by_code.get(code)
    if product:
        # Only write if at least one field changed
        changed = False
        for key, new_val in vals.items():
            old_val = product.get(key)
            if old_val != new_val:
                changed = True
                break
        if changed:
            odoo_json2('product.template', 'write', {'ids': [product['id']], 'vals': vals})
            updated_odoo_products += 1
        set_bexio_product_xml_id(product['id'], article['id'])
    else:
        new_id = odoo_json2('product.template', 'create', {'vals_list': [vals]})[0]
        set_bexio_product_xml_id(new_id, article['id'])
        created_odoo_products += 1
        odoo_by_code[code] = odoo_json2('product.template', 'read', {'ids': [new_id], 'fields': ODOO_PRODUCT_FIELDS})[0]

print(f'Bexio → Odoo products: {created_odoo_products} created, {updated_odoo_products} updated, {skipped_odoo_products} skipped.')


### Odoo → Bexio (Products)


In [ ]:
created_bexio_products = 0
updated_bexio_products = 0
skipped_bexio_products = 0

# Re-fetch Bexio articles to include any created in previous steps
bexio_articles_fresh = bexio_request('/article')
bexio_fresh_by_code = {}
for a in bexio_articles_fresh:
    code = (a.get('intern_code') or '').strip()
    if code:
        bexio_fresh_by_code[code] = a

for product in odoo_products:
    code = (product.get('default_code') or '').strip()
    if not code:
        skipped_bexio_products += 1
        continue

    # Delta sync: skip Odoo products not modified since last run
    if last_sync.get('odoo_products') and product.get('write_date'):
        if product['write_date'] <= last_sync['odoo_products']:
            continue

    vals = odoo_to_bexio_product_vals(product)
    if not vals.get('intern_name'):
        skipped_bexio_products += 1
        continue

    article = bexio_fresh_by_code.get(code)
    if article:
        bexio_request(f'/article/{article["id"]}', method='POST', data=vals)
        updated_bexio_products += 1
    else:
        # Bexio requires article_type_id on creation; fetch a default if needed
        article_types = bexio_request('/article_type')
        default_type = article_types[0]['id'] if article_types else 1
        vals['article_type_id'] = default_type
        vals['intern_code'] = code
        bexio_request('/article', method='POST', data=vals)
        created_bexio_products += 1

print(f'Odoo → Bexio products: {created_bexio_products} created, {updated_bexio_products} updated, {skipped_bexio_products} skipped.')


## Persist Product Sync Timestamps


In [ ]:
# Record max Bexio updated_at from after the product sync
bexio_articles_final = bexio_request('/article')
bexio_product_timestamps = [a.get('updated_at', '') for a in bexio_articles_final if a.get('updated_at')]
last_sync['bexio_products'] = max(bexio_product_timestamps) if bexio_product_timestamps else ''

# Record current UTC time for Odoo products
last_sync['odoo_products'] = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')

save_last_sync(last_sync)
print(f'Product sync timestamps saved: {last_sync}')


## Overall Sync Summary


In [ ]:
print('=' * 60)
print('OVERALL SYNC SUMMARY')
print('=' * 60)
print(f"{'Direction':<25} {'Created':>8} {'Updated':>8} {'Skipped':>8}")
print('-' * 60)
print(f"{'Contacts: Bexio → Odoo':<25} {created_odoo:>8} {updated_odoo:>8} {skipped_odoo:>8}")
print(f"{'Contacts: Odoo → Bexio':<25} {created_bexio:>8} {updated_bexio:>8} {skipped_bexio:>8}")
print(f"{'Products: Bexio → Odoo':<25} {created_odoo_products:>8} {updated_odoo_products:>8} {skipped_odoo_products:>8}")
print(f"{'Products: Odoo → Bexio':<25} {created_bexio_products:>8} {updated_bexio_products:>8} {skipped_bexio_products:>8}")
print('=' * 60)
print(f"Next run will use delta sync from {last_sync}")
